# Demo – Chatbot pháp luật (RAG)
Cần GPU để chạy LLM. Nếu không có GPU, đặt `LLM = "extractive"` để xem phần retrieval.

In [ ]:
# Settings -> Accelerator: GPU (T4 x2 or P100), Internet: ON
import os, subprocess
REPO = "https://github.com/dinhthithuthuyduphong-bot/LAB.git"
BRANCH = "RAG-based-chatbot-system"
WORK = "/kaggle/working/LAB" if os.path.exists("/kaggle") else "/content/LAB"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORK], check=True)
os.chdir(WORK)
# Large artifacts (chunks / BM25 / FAISS) can be kept in a Kaggle Dataset or Google Drive between sessions:
# os.environ["ARTIFACTS_DIR"] = "/kaggle/input/lab-rag-artifacts"   # read-only input -> copy if you need to write
!pip install -q -r requirements-gpu.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.exists("artifacts/bm25_legal"):
    !python -I scripts/build_corpus.py && python -I scripts/prepare_data.py && python -I scripts/build_chunks.py --strategy legal && python -I scripts/build_bm25.py --chunks legal

In [ ]:
import sys; sys.path.insert(0, "."); sys.path.insert(0, "app"); sys.path.insert(0, "scripts")
from demo import Bot, EXAMPLES
LLM = "Qwen/Qwen2.5-3B-Instruct"     # or "extractive"
RETRIEVER = "bm25_legal"             # "hybrid_legal_rerank" once the FAISS index is built
bot = Bot(RETRIEVER, LLM)

In [ ]:
from IPython.display import Markdown, display
for q in EXAMPLES:
    ans, src = bot.answer(q)
    display(Markdown(f"### ❓ {q}\n\n{ans}\n\n<details><summary>Nguồn</summary>\n\n{src}\n</details>"))

In [ ]:
# Interactive UI with a public link (Gradio, free)
!python -I app/demo.py --retriever {RETRIEVER} --llm {LLM}